In [703]:
#doing all the required imports
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

loading the datasets

In [704]:
claims = pd.read_csv("../data/raw/claims.csv")
policies = pd.read_csv("../data/raw/policies.csv")
garages = pd.read_csv("../data/raw/garages.csv")
adjusters = pd.read_csv("../data/raw/adjusters.csv")

In [705]:
claims.shape, policies.shape, garages.shape, adjusters.shape

((32396, 17), (26000, 10), (70, 6), (45, 4))

checking the quality of the data

In [706]:
claims.info()

<class 'pandas.DataFrame'>
RangeIndex: 32396 entries, 0 to 32395
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   claim_id              32396 non-null  str    
 1   policy_id             32396 non-null  str    
 2   garage_id             32396 non-null  str    
 3   adjuster_id           32396 non-null  str    
 4   incident_type         32396 non-null  str    
 5   incident_date         32396 non-null  str    
 6   incident_hour         32396 non-null  int64  
 7   reported_date         32396 non-null  str    
 8   claim_amount_xaf      32396 non-null  str    
 9   police_report         32396 non-null  str    
 10  witness_count         32396 non-null  int64  
 11  prior_claims_holder   32396 non-null  int64  
 12  vehicle_towed         32396 non-null  str    
 13  investigation_opened  32396 non-null  bool   
 14  days_to_settle        31845 non-null  float64
 15  amount_paid_xaf       32396 no

In [707]:
claims.isnull().sum()

claim_id                  0
policy_id                 0
garage_id                 0
adjuster_id               0
incident_type             0
incident_date             0
incident_hour             0
reported_date             0
claim_amount_xaf          0
police_report             0
witness_count             0
prior_claims_holder       0
vehicle_towed             0
investigation_opened      0
days_to_settle          551
amount_paid_xaf           0
fraud_flag                0
dtype: int64

In [708]:
claims.duplicated().sum()

np.int64(220)

In [709]:
claims["claim_id"].duplicated().sum()

np.int64(220)

In [710]:
duplicate_rows = claims[claims.duplicated(keep=False)]

duplicate_rows.groupby("claim_id").size().value_counts().sort_index()

2    220
Name: count, dtype: int64

In [711]:
#removing the duplicates from the claims.csv
claims = claims.drop_duplicates()

In [712]:
claims.shape

(32176, 17)

In [713]:
claims.duplicated().sum()

np.int64(0)

Checking the target

In [714]:
claims["fraud_flag"].value_counts()

fraud_flag
NO     31196
YES      980
Name: count, dtype: int64

In [715]:
claims["fraud_flag"].value_counts(normalize=True)

fraud_flag
NO     0.969543
YES    0.030457
Name: proportion, dtype: float64

In [716]:
#checking it again to see if the duplicated values are still there
claims.isnull().sum()

claim_id                  0
policy_id                 0
garage_id                 0
adjuster_id               0
incident_type             0
incident_date             0
incident_hour             0
reported_date             0
claim_amount_xaf          0
police_report             0
witness_count             0
prior_claims_holder       0
vehicle_towed             0
investigation_opened      0
days_to_settle          548
amount_paid_xaf           0
fraud_flag                0
dtype: int64

In [717]:
policies.isnull().sum()

policy_id             0
holder_id             0
region                0
vehicle_make          0
vehicle_year          0
cover_type            0
sum_insured_xaf       0
annual_premium_xaf    0
policy_start          0
payment_frequency     0
dtype: int64

In [718]:
garages.isnull().sum()

garage_id          0
garage_name        0
town               0
registered_year    0
bay_count          0
approved           0
dtype: int64

In [719]:
adjusters.isnull().sum()

adjuster_id      0
region           0
hired_year       0
caseload_band    0
dtype: int64

In [720]:
#checking the ids before merging to ensure that they are unique
claims["claim_id"].nunique(), policies["policy_id"].nunique(), garages["garage_id"].nunique(), adjusters["adjuster_id"].nunique()

(32176, 26000, 70, 45)

In [721]:
claims["policy_id"].isin(policies["policy_id"]).value_counts()

policy_id
True     32175
False        1
Name: count, dtype: int64

In [722]:
claims["garage_id"].isin(garages["garage_id"]).value_counts()

garage_id
True    32176
Name: count, dtype: int64

In [723]:
claims["adjuster_id"].isin(adjusters["adjuster_id"]).value_counts()

adjuster_id
True    32176
Name: count, dtype: int64

In [724]:
claims["policy_id"].isin(policies["policy_id"]).sum()

np.int64(32175)

Merging the datasets

In [725]:
model_data = claims.merge(
    policies,
    on="policy_id",
    how="left"
)

model_data = model_data.merge(
    garages,
    on="garage_id",
    how="left"
)

model_data = model_data.merge(
    adjusters,
    on="adjuster_id",
    how="left"
)

In [726]:
model_data.shape

(32176, 34)

In [727]:
model_data["claim_id"].nunique()

32176

In [728]:
model_data.columns.tolist()

['claim_id',
 'policy_id',
 'garage_id',
 'adjuster_id',
 'incident_type',
 'incident_date',
 'incident_hour',
 'reported_date',
 'claim_amount_xaf',
 'police_report',
 'witness_count',
 'prior_claims_holder',
 'vehicle_towed',
 'investigation_opened',
 'days_to_settle',
 'amount_paid_xaf',
 'fraud_flag',
 'holder_id',
 'region_x',
 'vehicle_make',
 'vehicle_year',
 'cover_type',
 'sum_insured_xaf',
 'annual_premium_xaf',
 'policy_start',
 'payment_frequency',
 'garage_name',
 'town',
 'registered_year',
 'bay_count',
 'approved',
 'region_y',
 'hired_year',
 'caseload_band']

In [729]:
# Parse dates used by feature engineering and temporal evaluation.

model_data["incident_date_clean"] = pd.to_datetime(
    model_data["incident_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

model_data["reported_date_clean"] = pd.to_datetime(
    model_data["reported_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

model_data["policy_start_clean"] = pd.to_datetime(
    model_data["policy_start"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

assert "incident_date_clean" in model_data.columns
assert "reported_date_clean" in model_data.columns
assert "policy_start_clean" in model_data.columns

In [730]:
model_data[
    [
        "incident_date_clean",
        "reported_date_clean",
        "policy_start_clean"
    ]
].head()

,incident_date_clean,reported_date_clean,policy_start_clean
0,2026-06-23,2026-07-23,2024-01-10
1,2023-05-10,2023-07-10,2023-03-09
2,2025-06-14,2025-06-28,2025-05-13
3,2025-11-24,2025-11-24,2024-07-02
4,2024-11-23,2024-11-23,2024-06-08


In [731]:
# Feature engineering

X = pd.DataFrame(index=model_data.index)

# Claim-level features
X["incident_type"] = model_data["incident_type"]
X["incident_hour"] = model_data["incident_hour"]
X["police_report"] = model_data["police_report"]
X["witness_count"] = model_data["witness_count"]
X["prior_claims_holder"] = model_data["prior_claims_holder"]
X["vehicle_towed"] = model_data["vehicle_towed"]

# Policy features
X["region"] = model_data["region_x"]
X["vehicle_make"] = model_data["vehicle_make"]
X["vehicle_year"] = model_data["vehicle_year"]
X["cover_type"] = model_data["cover_type"]
X["sum_insured_xaf"] = model_data["sum_insured_xaf"]
X["annual_premium_xaf"] = model_data["annual_premium_xaf"]
X["payment_frequency"] = model_data["payment_frequency"]

# Garage features
X["town"] = model_data["town"]
X["registered_year"] = model_data["registered_year"]
X["bay_count"] = model_data["bay_count"]
X["approved"] = model_data["approved"]

# Adjuster features
X["adjuster_region"] = model_data["region_y"]
X["hired_year"] = model_data["hired_year"]
X["caseload_band"] = model_data["caseload_band"]

# Claim amount
X["claim_amount_clean"] = (
    model_data["claim_amount_xaf"]
    .str.replace(",", "", regex=False)
    .str.replace("XAF", "", regex=False)
    .str.strip()
    .astype(float)
)

# Domain features
X["claim_to_insured_ratio"] = (
    X["claim_amount_clean"] / model_data["sum_insured_xaf"]
)

X["reporting_delay"] = (
    model_data["reported_date_clean"]
    - model_data["incident_date_clean"]
).dt.days

X["policy_age_days"] = (
    model_data["incident_date_clean"]
    - model_data["policy_start_clean"]
).dt.days

X["vehicle_age"] = (
    model_data["incident_date_clean"].dt.year
    - model_data["vehicle_year"]
)

# Time features
X["incident_month"] = model_data["incident_date_clean"].dt.month
X["incident_dayofweek"] = model_data["incident_date_clean"].dt.dayofweek

# Required night-hour feature
X["night_hour"] = (
    (X["incident_hour"] < 6)
    | (X["incident_hour"] >= 22)
)

# Data-quality indicators
X["late_report"] = X["reporting_delay"] > 7
X["negative_reporting_delay"] = X["reporting_delay"] < 0
X["negative_policy_age"] = X["policy_age_days"] < 0

In [732]:
model_data[
    [
        "reported_date_clean",
        "garage_id",
        "adjuster_id"
    ]
].isna().sum()

reported_date_clean    0
garage_id              0
adjuster_id            0
dtype: int64

In [733]:
model_data[
    [
        "garage_id",
        "adjuster_id"
    ]
].nunique()

garage_id      70
adjuster_id    45
dtype: int64

In [734]:
[
    column
    for column in X.columns
    if column in ["garage_id", "adjuster_id"]
]

[]

In [735]:
history_data = model_data.copy()

history_data = history_data.sort_values(
    ["holder_id", "reported_date_clean", "claim_id"]
).copy()

# Number of claims from strictly earlier reported dates.
history_data["holder_claim_count"] = (
    history_data.groupby("holder_id")["reported_date_clean"]
    .transform(
        lambda s: s.map(
            s.value_counts().sort_index().cumsum().shift(fill_value=0)
        )
    )
)

# First reported date strictly before the current reported date.
history_data["holder_first_prior_reported_date"] = (
    history_data.groupby("holder_id")["reported_date_clean"]
    .transform(
        lambda s: s.drop_duplicates().shift().reindex(
            s.factorize(sort=True)[0]
        ).values
    )
)

In [736]:
history_check = history_data.copy()

history_check.loc[
    history_check["holder_claim_count"] == 0,
    [
        "holder_id",
        "holder_claim_count",
    ],
]

,holder_id,holder_claim_count
16190,HL400000,0.0
30970,HL400001,0.0
6543,HL400004,0.0
18503,HL400005,0.0
2459,HL400007,0.0
...,...,...
5410,HL418992,0.0
8975,HL418993,0.0
31745,HL418996,0.0
2736,HL418997,0.0


In [737]:
history_check = history_data.copy()

history_check.loc[
    history_check["holder_claim_count"] == 0,
    [
        "holder_id",
        "claim_id",
        "reported_date_clean",
        "holder_first_prior_reported_date"
    
    ]
].head(20)

,holder_id,claim_id,reported_date_clean,holder_first_prior_reported_date
16190,HL400000,CLM-021620,2024-06-24,NaT
30970,HL400001,CLM-029268,2025-10-09,NaT
6543,HL400004,CLM-020978,2024-03-12,NaT
18503,HL400005,CLM-016994,2024-04-11,NaT
2459,HL400007,CLM-003777,2025-08-18,NaT
31827,HL400009,CLM-001389,2025-03-05,NaT
28733,HL400010,CLM-030063,2024-11-12,NaT
11327,HL400011,CLM-023953,2023-12-31,NaT
10342,HL400012,CLM-012642,2024-02-26,NaT
19359,HL400013,CLM-006386,2026-03-17,NaT


In [738]:
history_data = model_data.copy()

# Sort claims by holder and reported date.
history_data = history_data.sort_values(
    ["holder_id", "reported_date_clean", "claim_id"],
    na_position="last"
).copy()

# Count claims for each holder on each reported date.
claims_per_day = (
    history_data
    .groupby(
        ["holder_id", "reported_date_clean"],
        dropna=False
    )
    .size()
    .rename("claims_on_date")
    .reset_index()
)

# Build one row per holder and reported date.
holder_dates = (
    history_data[
        ["holder_id", "reported_date_clean"]
    ]
    .drop_duplicates()
    .sort_values(
        ["holder_id", "reported_date_clean"],
        na_position="last"
    )
    .copy()
)

holder_dates = holder_dates.merge(
    claims_per_day,
    on=["holder_id", "reported_date_clean"],
    how="left"
)

# Number of claims from strictly earlier reported dates.
holder_dates["holder_claim_count"] = (
    holder_dates
    .groupby("holder_id")["claims_on_date"]
    .cumsum()
    - holder_dates["claims_on_date"]
)

# Earliest reported date strictly before the current date.
holder_dates["holder_first_prior_reported_date"] = (
    holder_dates
    .groupby("holder_id")["reported_date_clean"]
    .shift(1)
)

# A missing current reported date cannot have valid time-based history.
missing_date = holder_dates["reported_date_clean"].isna()

holder_dates.loc[
    missing_date,
    [
        "holder_claim_count",
        "holder_first_prior_reported_date"
    ]
] = np.nan

# History duration.
holder_dates["holder_history_days"] = (
    holder_dates["reported_date_clean"]
    - holder_dates["holder_first_prior_reported_date"]
).dt.days

# Prior claims per day of available history.
holder_dates["holder_claim_frequency"] = (
    holder_dates["holder_claim_count"]
    / holder_dates["holder_history_days"].replace(0, np.nan)
)

# Remove any existing history features before merging.
history_data = history_data.drop(
    columns=[
        "holder_claim_count",
        "holder_first_prior_reported_date",
        "holder_history_days",
        "holder_claim_frequency"
    ],
    errors="ignore"
)

# Add the history features back to every claim.
history_data = history_data.merge(
    holder_dates[
        [
            "holder_id",
            "reported_date_clean",
            "holder_claim_count",
            "holder_first_prior_reported_date",
            "holder_history_days",
            "holder_claim_frequency"
        ]
    ],
    on=["holder_id", "reported_date_clean"],
    how="left"
)

# Restore original row order.
history_data = history_data.sort_index()

X["holder_claim_count"] = history_data["holder_claim_count"]
X["holder_history_days"] = history_data["holder_history_days"]
X["holder_claim_frequency"] = history_data["holder_claim_frequency"]

In [739]:
history_feature_columns = [
    "holder_claim_count",
    "holder_history_days",
    "holder_claim_frequency"
]

X_base = X.drop(
    columns=history_feature_columns,
    errors="ignore"
)

In [740]:
history_check = history_data.copy()

history_check[
    [
        "holder_claim_count",
        "holder_first_prior_reported_date",
        "holder_history_days",
        "holder_claim_frequency"
    ]
].isna().sum()

holder_claim_count                      1
holder_first_prior_reported_date    11309
holder_history_days                 11309
holder_claim_frequency              11309
dtype: int64

In [741]:
history_check.loc[
    history_check["holder_claim_count"].notna()
    & history_check["holder_first_prior_reported_date"].notna()
    & (
        history_check["holder_first_prior_reported_date"]
        >= history_check["reported_date_clean"]
    ),
    [
        "claim_id",
        "holder_id",
        "reported_date_clean",
        "holder_claim_count",
        "holder_first_prior_reported_date"
    ]
].head(20)

,claim_id,holder_id,reported_date_clean,holder_claim_count,holder_first_prior_reported_date


In [742]:
history_check.loc[
    history_check["holder_claim_count"] < 0,
    [
        "claim_id",
        "holder_id",
        "reported_date_clean",
        "holder_claim_count",
        "holder_first_prior_reported_date",
    ]
].head(20)

,claim_id,holder_id,reported_date_clean,holder_claim_count,holder_first_prior_reported_date


In [743]:
history_check[
    history_check["holder_claim_count"].isna()
][
    [
        "claim_id",
        "holder_id",
        "reported_date_clean",
        "holder_claim_count",
        "holder_first_prior_reported_date",
    ]
].head(20)

,claim_id,holder_id,reported_date_clean,holder_claim_count,holder_first_prior_reported_date
32175,CLM-900000,NaN,2026-01-06,NaN,NaT


In [744]:
history_check["holder_claim_count"].isna().sum()

np.int64(1)

In [745]:
history_check["holder_first_prior_reported_date"].isna().sum()

np.int64(11309)

In [746]:
history_check.loc[
    history_check["holder_claim_count"] == 0,
    [
        "claim_id",
        "holder_id",
        "reported_date_clean",
        "holder_claim_count",
        "holder_first_prior_reported_date"
    ]
].head(20)

,claim_id,holder_id,reported_date_clean,holder_claim_count,holder_first_prior_reported_date
0,CLM-021620,HL400000,2024-06-24,0.0,NaT
1,CLM-029268,HL400001,2025-10-09,0.0,NaT
3,CLM-020978,HL400004,2024-03-12,0.0,NaT
7,CLM-016994,HL400005,2024-04-11,0.0,NaT
14,CLM-003777,HL400007,2025-08-18,0.0,NaT
18,CLM-001389,HL400009,2025-03-05,0.0,NaT
19,CLM-030063,HL400010,2024-11-12,0.0,NaT
21,CLM-023953,HL400011,2023-12-31,0.0,NaT
25,CLM-012642,HL400012,2024-02-26,0.0,NaT
29,CLM-006386,HL400013,2026-03-17,0.0,NaT


In [747]:
same_day_history = history_check[
    history_check["holder_first_prior_reported_date"].notna()
    & (
        history_check["holder_first_prior_reported_date"]
        == history_check["reported_date_clean"]
    )
]

same_day_history[
    [
        "claim_id",
        "holder_id",
        "reported_date_clean",
        "holder_first_prior_reported_date",
        "holder_claim_count",
    ]
].head(20)

,claim_id,holder_id,reported_date_clean,holder_first_prior_reported_date,holder_claim_count


In [748]:
history_check[
    [
        "holder_claim_count",
        "holder_first_prior_reported_date"
    ]
].head(20)

,holder_claim_count,holder_first_prior_reported_date
0,0.0,NaT
1,0.0,NaT
2,1.0,2025-10-09
3,0.0,NaT
4,1.0,2024-03-12
5,2.0,2024-12-21
6,3.0,2025-07-31
7,0.0,NaT
8,1.0,2024-04-11
9,2.0,2025-04-12


In [749]:
history_check = history_data.copy()

# Claims with no prior claims must have no prior date.
assert (
    history_check.loc[
        history_check["holder_claim_count"] == 0,
        "holder_first_prior_reported_date"
    ].isna()
).all()

# Every prior date must be strictly earlier than the current claim.
has_prior_history = (
    history_check["holder_first_prior_reported_date"].notna()
)

assert (
    history_check.loc[
        has_prior_history,
        "holder_first_prior_reported_date"
    ]
    < history_check.loc[
        has_prior_history,
        "reported_date_clean"
    ]
).all()

# Prior claim counts must never be negative.
assert (
    history_check["holder_claim_count"].dropna() >= 0
).all()

# Claims with no prior history must have no history duration.
assert (
    history_check.loc[
        history_check["holder_claim_count"] == 0,
        "holder_history_days"
    ].isna()
).all()

"Historical feature timing checks passed."

'Historical feature timing checks passed.'

In [750]:
bad_history_rows = history_check[
    (history_check["holder_claim_count"] == 0)
    & (history_check["holder_first_prior_reported_date"].notna())
]

bad_history_rows[
    [
        "claim_id",
        "holder_id",
        "reported_date_clean",
        "holder_claim_count",
        "holder_first_prior_reported_date"
    ]
].head(20)

,claim_id,holder_id,reported_date_clean,holder_claim_count,holder_first_prior_reported_date


In [751]:
# Target

y = model_data["fraud_flag"].map({
    "NO": 0,
    "YES": 1
})

In [752]:
X.shape

(32176, 34)

In [753]:
X.columns.tolist()

['incident_type',
 'incident_hour',
 'police_report',
 'witness_count',
 'prior_claims_holder',
 'vehicle_towed',
 'region',
 'vehicle_make',
 'vehicle_year',
 'cover_type',
 'sum_insured_xaf',
 'annual_premium_xaf',
 'payment_frequency',
 'town',
 'registered_year',
 'bay_count',
 'approved',
 'adjuster_region',
 'hired_year',
 'caseload_band',
 'claim_amount_clean',
 'claim_to_insured_ratio',
 'reporting_delay',
 'policy_age_days',
 'vehicle_age',
 'incident_month',
 'incident_dayofweek',
 'night_hour',
 'late_report',
 'negative_reporting_delay',
 'negative_policy_age',
 'holder_claim_count',
 'holder_history_days',
 'holder_claim_frequency']

In [754]:
X.dtypes.value_counts()

str        11
float64    10
int64       7
bool        4
int32       2
Name: count, dtype: int64

In [755]:
post_assessment_columns = [
    "investigation_opened",
    "days_to_settle",
    "amount_paid_xaf",
    "fraud_flag",
]

leakage_present = [
    column
    for column in post_assessment_columns
    if column in X.columns
]

assert not leakage_present, (
    "Post-assessment columns reached the model matrix: "
    f"{leakage_present}"
)

leakage_present

[]

In [756]:
assert not leakage_present, (
    f"Post-assessment columns reached the model matrix: {leakage_present}"
)

In [757]:
history_feature_columns = [
    "holder_claim_count",
    "holder_history_days",
    "holder_claim_frequency"
]

X_base = X.drop(
    columns=history_feature_columns,
    errors="ignore"
)

In [758]:
X_train, X_test, y_train, y_test = train_test_split(
    X_base,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [759]:
X_train.index.intersection(X_test.index).size

0

In [760]:
X_train.index.equals(
    model_data.loc[X_train.index].index
), X_test.index.equals(
    model_data.loc[X_test.index].index
)

(True, True)

In [761]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((25740, 31), (6436, 31), (25740,), (6436,))

In [762]:
def add_holder_history_features_fast(scoring_data, history_source):
    scoring = scoring_data[
        ["holder_id", "reported_date_clean"]
    ].copy()

    source = history_source[
        ["holder_id", "reported_date_clean"]
    ].copy()

    source = source[
        source["reported_date_clean"].notna()
    ].copy()

    scoring["_original_index"] = scoring.index

    # Claims available in the history source by holder and date.
    source_counts = (
        source
        .groupby(
            ["holder_id", "reported_date_clean"]
        )
        .size()
        .reset_index(name="claims_on_date")
    )

    # Cumulative claims in the history source.
    source_counts = source_counts.sort_values(
        ["reported_date_clean", "holder_id"]
    )

    source_counts["cumulative_claims"] = (
        source_counts
        .groupby("holder_id")["claims_on_date"]
        .cumsum()
    )

    # Sort by date first because merge_asof requires the
    # merge key to be globally sorted.
    scoring_sorted = scoring.sort_values(
        ["reported_date_clean", "holder_id"]
    )

    source_sorted = source_counts.sort_values(
        ["reported_date_clean", "holder_id"]
    )

    # Find the most recent strictly earlier reported date.
    result = pd.merge_asof(
        scoring_sorted,
        source_sorted[
            [
                "holder_id",
                "reported_date_clean",
                "cumulative_claims"
            ]
        ],
        by="holder_id",
        on="reported_date_clean",
        direction="backward",
        allow_exact_matches=False
    )

    result["holder_claim_count"] = (
        result["cumulative_claims"].fillna(0)
    )

    # First reported date in the history source.
    first_dates = (
        source
        .groupby("holder_id")["reported_date_clean"]
        .min()
        .rename("holder_first_prior_reported_date")
        .reset_index()
    )

    result = result.merge(
        first_dates,
        on="holder_id",
        how="left"
    )

    result["holder_history_days"] = (
        result["reported_date_clean"]
        - result["holder_first_prior_reported_date"]
    ).dt.days

    # No prior claims means no history duration/frequency.
    result.loc[
        result["holder_claim_count"] == 0,
        [
            "holder_first_prior_reported_date",
            "holder_history_days"
        ]
    ] = [pd.NaT, np.nan]

    result["holder_claim_frequency"] = (
        result["holder_claim_count"]
        / result["holder_history_days"].replace(0, np.nan)
    )

    result.loc[
        result["reported_date_clean"].isna(),
        [
            "holder_claim_count",
            "holder_history_days",
            "holder_claim_frequency"
        ]
    ] = np.nan

    return result.set_index("_original_index")[
        [
            "holder_claim_count",
            "holder_first_prior_reported_date",
            "holder_history_days",
            "holder_claim_frequency"
        ]
    ]

In [763]:
def evaluate_cv_fast(
    X_data,
    y_data,
    model_pipeline,
    splitter,
    history_data,
    groups=None
):
    pr_auc_scores = []
    roc_auc_scores = []

    for train_idx, valid_idx in splitter.split(
        X_data,
        y_data,
        groups=groups
    ):
        X_fold_train = X_data.iloc[train_idx].copy()
        X_fold_valid = X_data.iloc[valid_idx].copy()

        y_fold_train = y_data.iloc[train_idx]
        y_fold_valid = y_data.iloc[valid_idx]

        train_indices = X_fold_train.index
        valid_indices = X_fold_valid.index

        train_history = add_holder_history_features_fast(
            history_data.loc[train_indices],
            history_data.loc[train_indices]
        )

        valid_history = add_holder_history_features_fast(
            history_data.loc[valid_indices],
            history_data.loc[train_indices]
        )

        for column in [
            "holder_claim_count",
            "holder_history_days",
            "holder_claim_frequency"
        ]:
            X_fold_train[column] = train_history[column]
            X_fold_valid[column] = valid_history[column]

        fold_model = clone(model_pipeline)

        fold_model.fit(
            X_fold_train,
            y_fold_train
        )

        probabilities = fold_model.predict_proba(
            X_fold_valid
        )[:, 1]

        pr_auc_scores.append(
            average_precision_score(
                y_fold_valid,
                probabilities
            )
        )

        roc_auc_scores.append(
            roc_auc_score(
                y_fold_valid,
                probabilities
            )
        )

    return {
        "pr_auc_scores": pr_auc_scores,
        "pr_auc_mean": np.mean(pr_auc_scores),
        "pr_auc_std": np.std(pr_auc_scores),
        "roc_auc_scores": roc_auc_scores,
        "roc_auc_mean": np.mean(roc_auc_scores),
        "roc_auc_std": np.std(roc_auc_scores)
    }

In [764]:
def make_logistic_pipeline(include_garage_name=False):
    numeric_columns = [
        "incident_hour",
        "witness_count",
        "prior_claims_holder",
        "vehicle_year",
        "sum_insured_xaf",
        "annual_premium_xaf",
        "registered_year",
        "bay_count",
        "hired_year",
        "claim_amount_clean",
        "claim_to_insured_ratio",
        "reporting_delay",
        "policy_age_days",
        "vehicle_age",
        "incident_month",
        "incident_dayofweek",
        "holder_claim_count",
        "holder_history_days",
        "holder_claim_frequency"
    ]

    categorical_columns = [
        "incident_type",
        "police_report",
        "vehicle_towed",
        "region",
        "vehicle_make",
        "cover_type",
        "payment_frequency",
        "town",
        "approved",
        "adjuster_region",
        "caseload_band",
        "night_hour",
        "late_report",
        "negative_reporting_delay",
        "negative_policy_age"
    ]

    if include_garage_name:
        categorical_columns.append("garage_name")

    numeric_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_preprocessor, numeric_columns),
        ("categorical", categorical_preprocessor, categorical_columns)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

In [765]:
logistic_pipeline = make_logistic_pipeline(
    include_garage_name=False
)

In [766]:
random_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_cv_results = evaluate_cv_fast(
    X_base,
    y,
    logistic_pipeline,
    random_cv,
    model_data
)

random_cv_results

{'pr_auc_scores': [0.10274379410286272,
  0.11820549574635471,
  0.13013906851617524,
  0.0898978805250392,
  0.09797512005649066],
 'pr_auc_mean': np.float64(0.10779227178938451),
 'pr_auc_std': np.float64(0.014490599649055035),
 'roc_auc_scores': [0.7745462127158556,
  0.7679605902306427,
  0.8045924091707526,
  0.7664869762618945,
  0.7602220724802182],
 'roc_auc_mean': np.float64(0.7747616521718728),
 'roc_auc_std': np.float64(0.015595156683015542)}

In [767]:
grouped_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grouped_cv_results = evaluate_cv_fast(
    X_base,
    y,
    logistic_pipeline,
    grouped_cv,
    model_data,
    groups=model_data.loc[X_base.index, "garage_id"]
)

grouped_cv_results

{'pr_auc_scores': [0.04881078197956233,
  0.03496558391987671,
  0.12443392111811599,
  0.03590033614426671,
  0.027894399839849725],
 'pr_auc_mean': np.float64(0.05440100460033429),
 'pr_auc_std': np.float64(0.035660819932280326),
 'roc_auc_scores': [0.6935349649771566,
  0.459493821931806,
  0.7430807648510878,
  0.5793377164849263,
  0.39471210088605313],
 'roc_auc_mean': np.float64(0.5740318738262059),
 'roc_auc_std': np.float64(0.1327864560990812)}

In [768]:
X_with_garage_name = X_base.copy()

X_with_garage_name["garage_name"] = model_data.loc[
    X_base.index,
    "garage_name"
]

X_without_garage_name = X_with_garage_name.drop(
    columns=["garage_name"]
)

In [769]:
logistic_pipeline = make_logistic_pipeline(
    include_garage_name=False
)

In [770]:
garage_pipeline = make_logistic_pipeline(
    include_garage_name=True
)

no_garage_pipeline = make_logistic_pipeline(
    include_garage_name=False
)

garage_cv = evaluate_cv_fast(
    X_with_garage_name,
    y,
    garage_pipeline,
    grouped_cv,
    model_data,
    groups=model_data.loc[
        X_with_garage_name.index,
        "garage_id"
    ]
)

no_garage_cv = evaluate_cv_fast(
    X_without_garage_name,
    y,
    no_garage_pipeline,
    grouped_cv,
    model_data, 
    groups=model_data.loc[
        X_without_garage_name.index,
        "garage_id"
    ]
)

garage_cv, no_garage_cv

({'pr_auc_scores': [0.04347782957161183,
   0.03574655488725163,
   0.11386769699798949,
   0.040792768191939825,
   0.03238128060640662],
  'pr_auc_mean': np.float64(0.05325322605103988),
  'pr_auc_std': np.float64(0.030551782653219377),
  'roc_auc_scores': [0.6606596014762317,
   0.47042896902226533,
   0.7398891088200902,
   0.6181674222986763,
   0.473253978887147],
  'roc_auc_mean': np.float64(0.5924798161008821),
  'roc_auc_std': np.float64(0.10597079348377346)},
 {'pr_auc_scores': [0.04881078197956233,
   0.03496558391987671,
   0.12443392111811599,
   0.03590033614426671,
   0.027894399839849725],
  'pr_auc_mean': np.float64(0.05440100460033429),
  'pr_auc_std': np.float64(0.035660819932280326),
  'roc_auc_scores': [0.6935349649771566,
   0.459493821931806,
   0.7430807648510878,
   0.5793377164849263,
   0.39471210088605313],
  'roc_auc_mean': np.float64(0.5740318738262059),
  'roc_auc_std': np.float64(0.1327864560990812)})

In [771]:
"garage_name" in X_base.columns

False

In [772]:
temporal_data = model_data.loc[X_base.index].copy()

temporal_data = temporal_data.sort_values(
    "reported_date_clean"
)

temporal_data["reported_date_clean"].min(), temporal_data["reported_date_clean"].max()

(Timestamp('2023-01-07 00:00:00'), Timestamp('2026-12-08 00:00:00'))

In [773]:
temporal_train = temporal_data[
    temporal_data["reported_date_clean"] < "2026-01-01"
].copy()

temporal_eval = temporal_data[
    (temporal_data["reported_date_clean"] >= "2026-01-01")
    & (temporal_data["reported_date_clean"] < "2026-07-01")
].copy()

temporal_test = temporal_data[
    temporal_data["reported_date_clean"] >= "2026-07-01"
].copy()

(
    temporal_train.shape,
    temporal_eval.shape,
    temporal_test.shape,
    temporal_train["reported_date_clean"].min(),
    temporal_train["reported_date_clean"].max(),
    temporal_eval["reported_date_clean"].min(),
    temporal_eval["reported_date_clean"].max(),
    temporal_test["reported_date_clean"].min(),
    temporal_test["reported_date_clean"].max()
)

((23378, 37),
 (7102, 37),
 (1696, 37),
 Timestamp('2023-01-07 00:00:00'),
 Timestamp('2025-12-31 00:00:00'),
 Timestamp('2026-01-01 00:00:00'),
 Timestamp('2026-06-30 00:00:00'),
 Timestamp('2026-07-01 00:00:00'),
 Timestamp('2026-12-08 00:00:00'))

In [774]:
X_temporal_train = X_base.loc[temporal_train.index].copy()
X_temporal_eval = X_base.loc[temporal_eval.index].copy()
X_temporal_test = X_base.loc[temporal_test.index].copy()

y_temporal_train = y.loc[temporal_train.index]
y_temporal_eval = y.loc[temporal_eval.index]
y_temporal_test = y.loc[temporal_test.index]

In [775]:
temporal_pipeline = make_logistic_pipeline(
    include_garage_name=False
)

In [776]:
train_temporal_history = add_holder_history_features_fast(
    model_data.loc[X_temporal_train.index],
    model_data.loc[X_temporal_train.index]
)

eval_temporal_history = add_holder_history_features_fast(
    model_data.loc[X_temporal_eval.index],
    model_data.loc[X_temporal_train.index]
)

test_temporal_history = add_holder_history_features_fast(
    model_data.loc[X_temporal_test.index],
    model_data.loc[
        X_temporal_train.index.union(
            X_temporal_eval.index
        )
    ]
)

In [777]:
history_columns = [
    "holder_claim_count",
    "holder_history_days",
    "holder_claim_frequency"
]

for column in history_columns:
    X_temporal_train[column] = train_temporal_history[column]
    X_temporal_eval[column] = eval_temporal_history[column]
    X_temporal_test[column] = test_temporal_history[column]

In [778]:
temporal_pipeline.fit(
    X_temporal_train,
    y_temporal_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](34,)","['incident_type','incident_hour','police_report',...,'holder_claim_count', 'holder_history_days','holder_claim_frequency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,34
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default

In [779]:
temporal_eval_probabilities = temporal_pipeline.predict_proba(
    X_temporal_eval
)[:, 1]

temporal_eval_pr_auc = average_precision_score(
    y_temporal_eval,
    temporal_eval_probabilities
)

temporal_eval_roc_auc = roc_auc_score(
    y_temporal_eval,
    temporal_eval_probabilities
)

temporal_eval_pr_auc, temporal_eval_roc_auc

(0.1142788506531339, 0.7538578471365117)

In [780]:
X_pre_holdout = X_base.loc[
    temporal_train.index.union(
        temporal_eval.index
    )
].copy()

y_pre_holdout = y.loc[
    X_pre_holdout.index
]

In [781]:
pre_holdout_history = add_holder_history_features_fast(
    model_data.loc[X_pre_holdout.index],
    model_data.loc[X_pre_holdout.index]
)

final_holdout_history = add_holder_history_features_fast(
    model_data.loc[X_temporal_test.index],
    model_data.loc[X_pre_holdout.index]
)

In [782]:
for column in history_columns:
    X_pre_holdout[column] = pre_holdout_history[column]
    X_temporal_test[column] = final_holdout_history[column]

In [783]:
final_temporal_pipeline = make_logistic_pipeline(
    include_garage_name=False
)

In [784]:
final_temporal_pipeline.fit(
    X_pre_holdout,
    y_pre_holdout
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](34,)","['incident_type','incident_hour','police_report',...,'holder_claim_count', 'holder_history_days','holder_claim_frequency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,34
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default

In [785]:
final_holdout_probabilities = final_temporal_pipeline.predict_proba(
    X_temporal_test
)[:, 1]

final_holdout_pr_auc = average_precision_score(
    y_temporal_test,
    final_holdout_probabilities
)

final_holdout_roc_auc = roc_auc_score(
    y_temporal_test,
    final_holdout_probabilities
)

final_holdout_pr_auc, final_holdout_roc_auc

(0.16693673864636507, 0.7949193369465309)

In [786]:
for name, data in [
    ("Development", temporal_train),
    ("Temporal evaluation", temporal_eval),
    ("Final holdout", temporal_test)
]:
    print(
        name,
        "rows =", len(data),
        "fraud rate =",
        data["fraud_flag"].eq("YES").mean()
    )

Development rows = 23378 fraud rate = 0.026905637779108562
Temporal evaluation rows = 7102 fraud rate = 0.0384398760912419
Final holdout rows = 1696 fraud rate = 0.045990566037735846


In [787]:
X_base.columns.tolist()

['incident_type',
 'incident_hour',
 'police_report',
 'witness_count',
 'prior_claims_holder',
 'vehicle_towed',
 'region',
 'vehicle_make',
 'vehicle_year',
 'cover_type',
 'sum_insured_xaf',
 'annual_premium_xaf',
 'payment_frequency',
 'town',
 'registered_year',
 'bay_count',
 'approved',
 'adjuster_region',
 'hired_year',
 'caseload_band',
 'claim_amount_clean',
 'claim_to_insured_ratio',
 'reporting_delay',
 'policy_age_days',
 'vehicle_age',
 'incident_month',
 'incident_dayofweek',
 'night_hour',
 'late_report',
 'negative_reporting_delay',
 'negative_policy_age']

In [788]:
X_base.shape, y.shape

((32176, 31), (32176,))

In [789]:
train_history = add_holder_history_features_fast(
    model_data.loc[X_train.index],
    model_data.loc[X_train.index]
)

test_history = add_holder_history_features_fast(
    model_data.loc[X_test.index],
    model_data.loc[X_train.index]
)

X_train = X_train.copy()
X_test = X_test.copy()

for column in [
    "holder_claim_count",
    "holder_history_days",
    "holder_claim_frequency"
]:
    X_train[column] = train_history[column]
    X_test[column] = test_history[column]

In [790]:
X_train[
    [
        "holder_claim_count",
        "holder_history_days",
        "holder_claim_frequency"
    ]
].head()

,holder_claim_count,holder_history_days,holder_claim_frequency
11741,0.0,NaN,NaN
28367,2.0,117.0,0.017094
28155,1.0,48.0,0.020833
15628,2.0,452.0,0.004425
4750,0.0,NaN,NaN


In [791]:
X_test[
    [
        "holder_claim_count",
        "holder_history_days",
        "holder_claim_frequency"
    ]
].head()

,holder_claim_count,holder_history_days,holder_claim_frequency
19143,2.0,253.0,0.007905
15299,0.0,NaN,NaN
7505,1.0,212.0,0.004717
13511,1.0,54.0,0.018519
16247,3.0,499.0,0.006012


In [792]:
# Validation rows must never be used as the history source.
validation_indices = set(X_test.index)
training_indices = set(X_train.index)

assert validation_indices.isdisjoint(training_indices)

# Validation history was calculated using the training set as its source.
# Therefore the validation history cannot contain validation-only claims.
test_history_check = add_holder_history_features_fast(
    model_data.loc[X_test.index],
    model_data.loc[X_train.index]
)

pd.testing.assert_series_equal(
    X_test["holder_claim_count"].sort_index(),
    test_history_check["holder_claim_count"].sort_index(),
    check_names=False
)

"Fold-safe holder history checks passed."

'Fold-safe holder history checks passed.'

Preprocessing

In [793]:

numeric_columns = X_train.select_dtypes(
    include=["int64", "float64", "bool"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_columns),
    ("categorical", categorical_preprocessor, categorical_columns)
])

/var/folders/yw/1_m9x15n30b4vv_cc8hvqcg80000gn/T/ipykernel_2798/3152382602.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


In [794]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [795]:
logistic_pipeline.fit(X_train, y_train)

logistic_test_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

logistic_test_pr_auc = average_precision_score(
    y_test,
    logistic_test_probabilities
)

logistic_test_roc_auc = roc_auc_score(
    y_test,
    logistic_test_probabilities
)

logistic_test_pr_auc, logistic_test_roc_auc

(0.11446782147240629, 0.7908277734170591)

In [796]:
logistic_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](34,)","['incident_type','incident_hour','police_report',...,'holder_claim_count', 'holder_history_days','holder_claim_frequency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,34
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default

In [797]:
logistic_pipeline.named_steps

{'preprocessor': ColumnTransformer(transformers=[('numeric',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median')),
                                                  ('scaler', StandardScaler())]),
                                  ['incident_hour', 'witness_count',
                                   'prior_claims_holder', 'vehicle_year',
                                   'sum_insured_xaf', 'annual_premium_xaf',
                                   'registered_year', 'bay_count', 'hired_year',
                                   'claim_amount_clean',
                                   'claim_to_insured_ratio', 'reporting_delay',...
                                   'negative_policy_age', 'holder_claim_count',
                                   'holder_history_days',
                                   'holder_claim_frequency']),
                                 ('categorical',
                   

In [798]:
forbidden_columns = [
    "investigation_opened",
    "days_to_settle",
    "amount_paid_xaf",
    "fraud_flag"
]

forbidden_in_X = [
    col for col in forbidden_columns
    if col in X.columns
]

forbidden_in_X

[]

In [799]:
numeric_columns, categorical_columns

(['incident_hour',
  'witness_count',
  'prior_claims_holder',
  'vehicle_year',
  'sum_insured_xaf',
  'annual_premium_xaf',
  'registered_year',
  'bay_count',
  'hired_year',
  'claim_amount_clean',
  'claim_to_insured_ratio',
  'reporting_delay',
  'policy_age_days',
  'vehicle_age',
  'night_hour',
  'late_report',
  'negative_reporting_delay',
  'negative_policy_age',
  'holder_claim_count',
  'holder_history_days',
  'holder_claim_frequency'],
 ['incident_type',
  'police_report',
  'vehicle_towed',
  'region',
  'vehicle_make',
  'cover_type',
  'payment_frequency',
  'town',
  'approved',
  'adjuster_region',
  'caseload_band'])

In [800]:
final_feature_columns = X.columns.tolist()

len(final_feature_columns), final_feature_columns


(34,
 ['incident_type',
  'incident_hour',
  'police_report',
  'witness_count',
  'prior_claims_holder',
  'vehicle_towed',
  'region',
  'vehicle_make',
  'vehicle_year',
  'cover_type',
  'sum_insured_xaf',
  'annual_premium_xaf',
  'payment_frequency',
  'town',
  'registered_year',
  'bay_count',
  'approved',
  'adjuster_region',
  'hired_year',
  'caseload_band',
  'claim_amount_clean',
  'claim_to_insured_ratio',
  'reporting_delay',
  'policy_age_days',
  'vehicle_age',
  'incident_month',
  'incident_dayofweek',
  'night_hour',
  'late_report',
  'negative_reporting_delay',
  'negative_policy_age',
  'holder_claim_count',
  'holder_history_days',
  'holder_claim_frequency'])

In [801]:
required_features = [
    "claim_amount_clean",
    "claim_to_insured_ratio",
    "reporting_delay",
    "policy_age_days",
    "night_hour",
    "holder_claim_count",
    "holder_history_days",
    "holder_claim_frequency"
]

missing_required_features = [
    column for column in required_features
    if column not in final_feature_columns
]

missing_required_features

[]

In [802]:
final_feature_contract = pd.DataFrame({
    "feature": final_feature_columns,
    "type": [
        "numeric" if feature in numeric_columns else "categorical"
        for feature in final_feature_columns
    ]
})

final_feature_contract

,feature,type
0,incident_type,categorical
1,incident_hour,numeric
2,police_report,categorical
3,witness_count,numeric
4,prior_claims_holder,numeric
5,vehicle_towed,categorical
6,region,categorical
7,vehicle_make,categorical
8,vehicle_year,numeric
9,cover_type,categorical


In [803]:
test_predictions = logistic_pipeline.predict(X_test)
test_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

len(test_predictions), len(test_probabilities)

(6436, 6436)

In [804]:
average_precision_score(y_test, test_probabilities)

0.11446782147240629

In [805]:
assert "preprocessor" in logistic_pipeline.named_steps
assert "model" in logistic_pipeline.named_steps

assert not any(
    column in final_feature_columns
    for column in [
        "investigation_opened",
        "days_to_settle",
        "amount_paid_xaf",
        "fraud_flag"
    ]
)

print("Rochelle's feature pipeline checks passed.")

Rochelle's feature pipeline checks passed.
